# DemandLens sales analysis

An executed analysis of real UCI Online Retail transactions using the shared preparation pipeline. Forecast target: positive recorded units after unambiguous same-day reversals, not unconstrained demand. Source: Chen (2015), https://doi.org/10.24432/C5BW33, CC BY 4.0.

Run prepare, evaluate, and publish-demo before rerunning this notebook.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
import plotly.express as px
ROOT = Path.cwd() if (Path.cwd() / 'demandlens').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from demandlens.config import PROCESSED, DEMO, ARTIFACTS
from demandlens.queries import query
from demandlens.storage import load_bundle
manifest = load_bundle()
audit = json.loads((ARTIFACTS / 'cleaning_report.json').read_text())
sales = pd.read_parquet(PROCESSED / 'daily_sales.parquet')
retail = pd.read_parquet(PROCESSED / 'retail_daily.parquet')
products = pd.read_parquet(PROCESSED / 'products.parquet')
print(f"Source rows: {audit['raw_rows']:,}; retained positive sales rows: {audit['retained_sales_rows']:,}")
print(f"Products: {len(products)}; daily rows: {len(sales):,}; history: {sales.date.min().date()} to {sales.date.max().date()}")


Source rows: 541,909; retained positive sales rows: 525,712
Products: 20; daily rows: 7,451; history: 2010-12-01 to 2011-12-08


## Missingness and exclusions

Missing customer IDs do not remove valid sales. Counts below are measured, not copied from repository metadata. Exclusions reconcile sequentially.

In [2]:
print(pd.Series(audit['missing_values'], name='missing').to_string())
print(pd.DataFrame(audit['rules']).to_string(index=False))
print('Duplicate-looking source rows (retained unless another rule excludes them):', audit['duplicate_looking_rows_retained_by_policy'])
print('Training-only 99.5th-percentile quantity threshold:', audit['large_order_threshold_training_p995'])
print('Flagged large-order lines retained:', audit['large_sales_rows_retained'])
print('Unique full same-day reversal pairs:', audit['matched_reversal_pairs'])
print('Merchandise units removed before final boundary:', audit['matched_merchandise_units_removed'])


Country             0
CustomerID     135080
Description      1454
InvoiceDate         0
InvoiceNo           0
Quantity            0
StockCode           0
UnitPrice           0
                              rule  affected_rows  before  after
           invalid_required_values              0  541909 541909
          returns_or_cancellations          10624  541909 531285
         matched_same_day_reversal            454  531285 530831
zero_quantity_or_nonpositive_price           1181  530831 529650
                   non_merchandise           2321  529650 527329
               final_recorded_date           1617  527329 525712
Duplicate-looking source rows (retained unless another rule excludes them): 5268
Training-only 99.5th-percentile quantity threshold: 192.0
Flagged large-order lines retained: 2167
Unique full same-day reversal pairs: 454
Merchandise units removed before final boundary: 85766.0


## Retail sales and returns

The chart uses all cleaned merchandise sales. Unique same-day full reversals remove the original purchase; other return quantities remain separate.

In [3]:
fig = px.line(retail, x='date', y='units_sold', title='All cleaned retail recorded units')
returns = pd.read_parquet(PROCESSED / 'returns_daily.parquet')
print('Absolute returned/cancelled units:', returns.return_units.sum())
print('Selected cohort share of cleaned units:', round(audit['selected_units_share'], 4))
fig


Absolute returned/cancelled units: 397958
Selected cohort share of cleaned units: 0.1038


Plotly figure

## Cohort and concentration

Eligibility and names are frozen through 31 August 2011. This is an established-product cohort, not all products.

In [4]:
print(products[['product_id','description','training_units','positive_sales_days','history_days']].to_string(index=False))
top = query(DEMO / 'demandlens.sqlite', 'top_products')
fig = px.bar(top.head(10), x='product_id', y='units_sold', title='Top ten products within selected cohort')
fig


product_id                         description  training_units  positive_sales_days  history_days
     84077   WORLD WAR 2 GLIDERS ASSTD DESIGNS           37378                  156           273
    85099B             JUMBO BAG RED RETROSPOT           31646                  214           274
     22197                SMALL POPCORN HOLDER           28141                  208           274
     21212     PACK OF 72 RETROSPOT CAKE CASES           27806                  214           274
    85123A  WHITE HANGING HEART T-LIGHT HOLDER           27768                  219           274
     84879       ASSORTED COLOUR BIRD ORNAMENT           24048                  214           274
     15036           ASSORTED COLOURS SILK FAN           21411                  171           273
     21977  PACK OF 60 PINK PAISLEY CAKE CASES           19488                  195           274
     22616           PACK OF 12 LONDON TISSUES           19377                  169           274
     17003          

Plotly figure

## Weekday patterns, zero-sales days, and variability

Inserted calendar dates are zero recorded sales. The data does not establish store closures or stock availability.

In [5]:
weekday = query(DEMO / 'demandlens.sqlite', 'weekday_sales')
fig = px.bar(weekday, x='weekday', y='average_units', title='Average recorded units; Monday = 0')
print(query(DEMO / 'demandlens.sqlite', 'zero_sales').to_string(index=False))
print(query(DEMO / 'demandlens.sqlite', 'variability').to_string(index=False))
fig


product_id  days  zero_days  zero_percent
     15036   372        141     37.903226
     17003   373        219     58.713137
     20725   373         76     20.375335
     21212   373         76     20.375335
     21977   373         97     26.005362
     22151   373        175     46.916890
     22178   373         84     22.520107
     22197   373         80     21.447721
     22386   373         86     23.056300
     22492   373        165     44.235925
     22616   373        135     36.193029
     22693   368        194     52.717391
     47566   371         78     21.024259
     84077   372        138     37.096774
     84755   373        126     33.780161
     84879   373         74     19.839142
     84991   373        107     28.686327
    85099B   373         74     19.839142
    85099F   373        111     29.758713
    85123A   373         69     18.498660
product_id  mean_units  variance_units  maximum_units
     15036   64.045699    36693.855438         1846.0
     17003

Plotly figure

## Period changes and large-order spikes

Differences are descriptive associations. Price changes, promotions, and stockouts are not identified as causes.

In [6]:
period = sales.assign(period=sales.date.map(lambda d: 'development' if d <= pd.Timestamp('2011-08-31') else 'validation' if d <= pd.Timestamp('2011-10-05') else 'test_and_final_history'))
print(period.groupby('period').agg(days=('date','nunique'), units=('units_sold','sum'), mean_daily_product_units=('units_sold','mean')).to_string())
print(sales.loc[sales.large_order_rows.gt(0)].nlargest(10, 'units_sold')[['product_id','date','units_sold','large_order_rows']].to_string(index=False))


                        days     units  mean_daily_product_units
period                                                          
development              274  395274.0                 72.248949
test_and_final_history    64  113719.0                 88.842969
validation                35   51331.0                 73.330000
product_id       date  units_sold  large_order_rows
     84077 2011-10-27      4848.0               1.0
     22197 2011-05-27      4314.0               1.0
     84879 2011-08-04      3359.0               1.0
     84077 2010-12-02      3264.0               2.0
    85123A 2011-01-11      3114.0               2.0
     22197 2011-12-08      3111.0               2.0
     21977 2011-05-23      2724.0               1.0
     84077 2011-02-03      2640.0               1.0
     22693 2010-12-16      2497.0               1.0
     22693 2011-02-11      2400.0               1.0


## Revised retrospective results

Validation selects the model within this revision. Changes were informed by the earlier test audit, so these results are not a fresh untouched holdout. Zero actual totals produce unavailable WAPE.

In [7]:
print('Validation-selected champion:', manifest['champion'])
print(query(DEMO / 'demandlens.sqlite', 'model_metrics').to_string(index=False))
print(query(DEMO / 'demandlens.sqlite', 'worst_predictions', model=manifest['champion']).head(3).to_string(index=False))


Validation-selected champion: trailing_median
          model  observations       mae       wape       bias
trailing_median          1260 71.899603  82.837431 -44.287698
         hgb_15          1260 77.279351  89.035580  -4.434113
  trailing_mean          1260 82.231831  94.741464  -1.882738
  weekday_mean4          1260 84.858532  97.767755  -1.882738
 seasonal_naive          1260 95.794444 110.367309   0.515079
product_id           model     origin target_date  horizon  actual  prediction  absolute_error
     84077 trailing_median 2011-10-27  2011-10-27        1  4848.0       109.5          4738.5
     22197 trailing_median 2011-10-27  2011-10-28        2  2065.0       125.0          1940.0
     22197 trailing_median 2011-11-03  2011-11-07        5  2012.0       135.0          1877.0


## Interpretation

One historical gift retailer and a development-selected cohort limit generalization. Valid large orders remain and can drive large errors. Forecasts are not calibrated intervals. Inventory scenarios use supplied stock and lead-time assumptions, with no measured savings claims. See the generated experiment report for the measured comparison and three observed failure cases.